In [18]:
import pandas as pd
import numpy as np
import datatable as dt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from statistics import mean
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [2]:
phenotype_df = pd.read_csv('/home/student2024/Desktop/Project_Depression/combined_dataset_phenotypes_corrected.csv')
phenotype_df.head()

,ID,Sex,Age,Depression,Study,depr_one_hot
0,GSM3562834_3999840035_R01C01,0,73,Control,1,0
1,GSM3562835_3999840035_R01C02,1,32,Case,1,1
2,GSM3562836_3999840035_R02C01,0,51,Case,1,1
3,GSM3562837_3999840035_R02C02,1,30,Case,1,1
4,GSM3562838_3999840035_R03C01,0,53,Case,1,1


In [3]:
mval = dt.fread('/home/student2024/Desktop/Project_Depression/Combined_Mval_non_adj.csv')
mval = mval.to_pandas()
#mval = mval.rename(columns={'C0':'ID'})
mval = mval.T
mval.columns = mval.iloc[0]
mval = mval[1:]
mval.head()

C0,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,cg00008004,...,cg27612019,cg27614432,cg27634071,cg27634744,cg27648216,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648
GSM3562834_3999840035_R01C01,3.699859,-4.60819,-5.758384,-3.428361,-1.607677,-6.670791,-3.107789,5.820279,-5.475742,-3.163562,...,0.402738,-4.935797,-5.004572,3.629387,3.625561,3.712343,2.162832,-3.205565,-4.658252,1.941852
GSM3562835_3999840035_R01C02,3.829019,-5.517733,-5.364797,-3.806508,-3.06947,-6.675133,-2.910615,6.075227,-5.647746,-2.860776,...,0.730073,-4.556461,-4.425977,3.305114,3.534774,3.963833,3.127694,-3.446988,-4.824407,1.924064
GSM3562836_3999840035_R02C01,4.543859,-5.298929,-5.672585,-3.913202,-2.032083,-6.644306,-4.005748,6.224032,-5.629312,-3.375355,...,0.916134,-4.753392,-5.297943,3.110546,4.186899,3.808285,3.060312,-2.811014,-4.545404,2.232796
GSM3562837_3999840035_R02C02,3.935193,-4.879178,-5.518779,-3.143026,-1.970269,-6.487348,-3.460816,5.762759,-5.47399,-3.410145,...,0.241682,-4.635074,-4.702293,2.967417,3.834561,3.631687,3.381633,-3.322698,-4.960753,2.02318
GSM3562838_3999840035_R03C01,4.207299,-4.579331,-5.579593,-3.098945,-1.041133,-6.402856,-2.857366,5.775951,-5.572881,-2.750484,...,0.428073,-5.145249,-5.329275,2.833336,3.631742,3.819408,3.013578,-3.525631,-4.398742,2.936999


In [4]:
mval.index.name = 'ID'
mval.reset_index(inplace=True)
mval.head()

C0,ID,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,...,cg27612019,cg27614432,cg27634071,cg27634744,cg27648216,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648
0,GSM3562834_3999840035_R01C01,3.699859,-4.60819,-5.758384,-3.428361,-1.607677,-6.670791,-3.107789,5.820279,-5.475742,...,0.402738,-4.935797,-5.004572,3.629387,3.625561,3.712343,2.162832,-3.205565,-4.658252,1.941852
1,GSM3562835_3999840035_R01C02,3.829019,-5.517733,-5.364797,-3.806508,-3.06947,-6.675133,-2.910615,6.075227,-5.647746,...,0.730073,-4.556461,-4.425977,3.305114,3.534774,3.963833,3.127694,-3.446988,-4.824407,1.924064
2,GSM3562836_3999840035_R02C01,4.543859,-5.298929,-5.672585,-3.913202,-2.032083,-6.644306,-4.005748,6.224032,-5.629312,...,0.916134,-4.753392,-5.297943,3.110546,4.186899,3.808285,3.060312,-2.811014,-4.545404,2.232796
3,GSM3562837_3999840035_R02C02,3.935193,-4.879178,-5.518779,-3.143026,-1.970269,-6.487348,-3.460816,5.762759,-5.47399,...,0.241682,-4.635074,-4.702293,2.967417,3.834561,3.631687,3.381633,-3.322698,-4.960753,2.02318
4,GSM3562838_3999840035_R03C01,4.207299,-4.579331,-5.579593,-3.098945,-1.041133,-6.402856,-2.857366,5.775951,-5.572881,...,0.428073,-5.145249,-5.329275,2.833336,3.631742,3.819408,3.013578,-3.525631,-4.398742,2.936999


In [5]:
combined_df = mval.merge(phenotype_df, how='outer', on='ID')
combined_df.head()

,ID,cg00000957,cg00001583,cg00002028,cg00002719,cg00002837,cg00003202,cg00003287,cg00007036,cg00007898,...,cg27656573,cg27657363,cg27657537,cg27662611,cg27665648,Sex,Age,Depression,Study,depr_one_hot
0,GSM3562834_3999840035_R01C01,3.699859,-4.60819,-5.758384,-3.428361,-1.607677,-6.670791,-3.107789,5.820279,-5.475742,...,3.712343,2.162832,-3.205565,-4.658252,1.941852,0,73,Control,1,0
1,GSM3562835_3999840035_R01C02,3.829019,-5.517733,-5.364797,-3.806508,-3.06947,-6.675133,-2.910615,6.075227,-5.647746,...,3.963833,3.127694,-3.446988,-4.824407,1.924064,1,32,Case,1,1
2,GSM3562836_3999840035_R02C01,4.543859,-5.298929,-5.672585,-3.913202,-2.032083,-6.644306,-4.005748,6.224032,-5.629312,...,3.808285,3.060312,-2.811014,-4.545404,2.232796,0,51,Case,1,1
3,GSM3562837_3999840035_R02C02,3.935193,-4.879178,-5.518779,-3.143026,-1.970269,-6.487348,-3.460816,5.762759,-5.47399,...,3.631687,3.381633,-3.322698,-4.960753,2.02318,1,30,Case,1,1
4,GSM3562838_3999840035_R03C01,4.207299,-4.579331,-5.579593,-3.098945,-1.041133,-6.402856,-2.857366,5.775951,-5.572881,...,3.819408,3.013578,-3.525631,-4.398742,2.936999,0,53,Case,1,1


In [6]:
combined_df.shape

(1630, 391216)

In [7]:
combined_df = combined_df.drop(columns=['ID','Depression'])

In [8]:
lr = LogisticRegression(max_iter = 10000)
rf = RandomForestClassifier()

In [9]:
sgkf = StratifiedGroupKFold(shuffle = True)
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []

In [11]:
x = combined_df.drop(columns=['Study','depr_one_hot'])
x = x.replace(-np.inf, pd.NA)
x = x.dropna(axis = 1)
y = combined_df['depr_one_hot']
groups = combined_df['Study']

In [12]:
for train_index, test_index in sgkf.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [13]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  61.943238843235385 % 
	 Mean ROC AUC:  51.1610051738373 % 

Logistic Regression: 
	MeanAccuracy:  64.75238214989687 % 
	 Mean ROC AUC:  50.46385427310991 %


Random Forest: 
	Mean Acuracy:  61.943238843235385 % 
	 Mean ROC AUC:  51.1610051738373 % 

Logistic Regression: 
	MeanAccuracy:  64.75238214989687 % 
	 Mean ROC AUC:  50.46385427310991 %


In [15]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.8513761467889911, 0.7961805555555553, 0.5290123456790125, 0.4744245524296676, 0.4461683417085426] 
ROC AUC:  [0.5002184139784946, 0.4984782608695653, 0.5443080357142857, 0.5031773743016759, 0.5118681738278437]
LR Accuracy:  [0.8532110091743119, 0.7986111111111112, 0.4444444444444444, 0.46547314578005117, 0.6758793969849246] 
 ROC AUC:  [0.5, 0.5, 0.48660714285714285, 0.5022926109412881, 0.5342929598570647]


RF Accuracy:  [0.8513761467889911, 0.7961805555555553, 0.5290123456790125, 0.4744245524296676, 0.4461683417085426] 
ROC AUC:  [0.5002184139784946, 0.4984782608695653, 0.5443080357142857, 0.5031773743016759, 0.5118681738278437]
LR Accuracy:  [0.8532110091743119, 0.7986111111111112, 0.4444444444444444, 0.46547314578005117, 0.6758793969849246] 
 ROC AUC:  [0.5, 0.5, 0.48660714285714285, 0.5022926109412881, 0.5342929598570647]

In [19]:
sss = StratifiedShuffleSplit()
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []
for train_index, test_index in sss.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [20]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  67.12576687116564 % 
	 Mean ROC AUC:  56.83619528619529 % 

Logistic Regression: 
	MeanAccuracy:  67.05521472392638 % 
	 Mean ROC AUC:  57.87373737373738 %


Random Forest: 
	Mean Acuracy:  67.12576687116564 % 
	 Mean ROC AUC:  56.83619528619529 % 

Logistic Regression: 
	MeanAccuracy:  67.05521472392638 % 
	 Mean ROC AUC:  57.87373737373738 %

In [21]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.6561349693251534, 0.66840490797546, 0.6588957055214724, 0.6641104294478526, 0.670245398773006, 0.6687116564417177, 0.6923312883435583, 0.6469325153374235, 0.6769938650306748, 0.7098159509202454] 
ROC AUC:  [0.5502356902356903, 0.5494570707070707, 0.5532112794612795, 0.5756607744107743, 0.5646759259259261, 0.5606186868686869, 0.5771043771043771, 0.5577904040404043, 0.5809217171717173, 0.6139436026936026]
LR Accuracy:  [0.656441717791411, 0.7055214723926381, 0.6687116564417178, 0.6441717791411042, 0.6687116564417178, 0.6748466257668712, 0.6748466257668712, 0.6319018404907976, 0.6503067484662577, 0.7300613496932515] 
 ROC AUC:  [0.5667508417508418, 0.6171717171717171, 0.5626262626262626, 0.5664141414141414, 0.576010101010101, 0.5761784511784511, 0.5672558922558922, 0.5482323232323232, 0.5487373737373737, 0.657996632996633]


RF Accuracy:  [0.6561349693251534, 0.66840490797546, 0.6588957055214724, 0.6641104294478526, 0.670245398773006, 0.6687116564417177, 0.6923312883435583, 0.6469325153374235, 0.6769938650306748, 0.7098159509202454] 
ROC AUC:  [0.5502356902356903, 0.5494570707070707, 0.5532112794612795, 0.5756607744107743, 0.5646759259259261, 0.5606186868686869, 0.5771043771043771, 0.5577904040404043, 0.5809217171717173, 0.6139436026936026]
LR Accuracy:  [0.656441717791411, 0.7055214723926381, 0.6687116564417178, 0.6441717791411042, 0.6687116564417178, 0.6748466257668712, 0.6748466257668712, 0.6319018404907976, 0.6503067484662577, 0.7300613496932515] 
 ROC AUC:  [0.5667508417508418, 0.6171717171717171, 0.5626262626262626, 0.5664141414141414, 0.576010101010101, 0.5761784511784511, 0.5672558922558922, 0.5482323232323232, 0.5487373737373737, 0.657996632996633]

In [25]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold()
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []
for train_index, test_index in skf.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

/home/student2024/anaconda3/lib/python3.11/site-packages/daal4py/sklearn/linear_model/logistic_path.py:574: ConvergenceWarning: lbfgs failed to converge (status=2):
ABNORMAL_TERMINATION_IN_LNSRCH.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [26]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  63.90797546012269 % 
	 Mean ROC AUC:  54.887013191641046 % 

Logistic Regression: 
	MeanAccuracy:  61.963190184049076 % 
	 Mean ROC AUC:  53.61337771377725 %


Random Forest: 
	Mean Acuracy:  63.90797546012269 % 
	 Mean ROC AUC:  54.887013191641046 % 

Logistic Regression: 
	MeanAccuracy:  61.963190184049076 % 
	 Mean ROC AUC:  53.61337771377725 %

In [27]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.6407975460122699, 0.6651840490797545, 0.5240797546012269, 0.6769938650306748, 0.6883435582822084] 
ROC AUC:  [0.4956102693602694, 0.5053135521885521, 0.6150012683380544, 0.5832938739271973, 0.5451316957679787]
LR Accuracy:  [0.6687116564417178, 0.6687116564417178, 0.48773006134969327, 0.6104294478527608, 0.6625766871165644] 
 ROC AUC:  [0.509090909090909, 0.5202441077441078, 0.580962245803915, 0.5703927620175031, 0.49997886103242717]


RF Accuracy:  [0.6407975460122699, 0.6651840490797545, 0.5240797546012269, 0.6769938650306748, 0.6883435582822084] 
ROC AUC:  [0.4956102693602694, 0.5053135521885521, 0.6150012683380544, 0.5832938739271973, 0.5451316957679787]
LR Accuracy:  [0.6687116564417178, 0.6687116564417178, 0.48773006134969327, 0.6104294478527608, 0.6625766871165644] 
 ROC AUC:  [0.509090909090909, 0.5202441077441078, 0.580962245803915, 0.5703927620175031, 0.49997886103242717]

In [28]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit()
lr_accu = []
lr_auc = []
rf_mean_accu = []
rf_mean_auc = []
for train_index, test_index in gss.split(x, y, groups):
    x_train_fold, x_test_fold = x.iloc[train_index], x.iloc[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]
    lr.fit(x_train_fold, y_train_fold)
    lr_pred = lr.predict(x_test_fold)
    lr_accu.append(accuracy_score(y_test_fold, lr_pred))
    lr_auc.append(roc_auc_score(y_test_fold, lr_pred))
    rf_accu = 0
    rf_auc = 0
    for i in range(20):
        rf.fit(x_train_fold, y_train_fold)
        rf_pred = rf.predict(x_test_fold)
        rf_accu += accuracy_score(y_test_fold, rf_pred)
        rf_auc += roc_auc_score(y_test_fold, rf_pred)
    rf_mean_accu.append(rf_accu/20)
    rf_mean_auc.append(rf_auc/20)

In [29]:
print('Random Forest: \n\tMean Acuracy: ', mean(rf_mean_accu)*100, '% \n\t Mean ROC AUC: ', mean(rf_mean_auc)*100, '% \n')
print('Logistic Regression: \n\tMeanAccuracy: ', mean(lr_accu)*100, '% \n\t Mean ROC AUC: ', mean(lr_auc)*100, '%')

Random Forest: 
	Mean Acuracy:  56.602614892984626 % 
	 Mean ROC AUC:  56.16461026065094 % 

Logistic Regression: 
	MeanAccuracy:  56.70207512025852 % 
	 Mean ROC AUC:  51.75182084831709 %


Random Forest: 
	Mean Acuracy:  56.602614892984626 % 
	 Mean ROC AUC:  56.16461026065094 % 

Logistic Regression: 
	MeanAccuracy:  56.70207512025852 % 
	 Mean ROC AUC:  51.75182084831709 %


In [30]:
print('RF Accuracy: ',rf_mean_accu,'\nROC AUC: ',rf_mean_auc)
print('LR Accuracy: ', lr_accu,'\n ROC AUC: ',lr_auc)

RF Accuracy:  [0.556822429906542, 0.7520634920634924, 0.4859110169491526, 0.4841188524590163, 0.551214953271028] 
ROC AUC:  [0.5978335979769465, 0.6130784574468084, 0.4949219873454127, 0.503201427920401, 0.5991950423429782]
LR Accuracy:  [0.5869158878504673, 0.7428571428571429, 0.4491525423728814, 0.4692622950819672, 0.5869158878504673] 
 ROC AUC:  [0.5168636791343213, 0.6009308510638298, 0.4522576934138625, 0.5006751396695192, 0.5168636791343213]


RF Accuracy:  [0.556822429906542, 0.7520634920634924, 0.4859110169491526, 0.4841188524590163, 0.551214953271028] 
ROC AUC:  [0.5978335979769465, 0.6130784574468084, 0.4949219873454127, 0.503201427920401, 0.5991950423429782]
LR Accuracy:  [0.5869158878504673, 0.7428571428571429, 0.4491525423728814, 0.4692622950819672, 0.5869158878504673] 
 ROC AUC:  [0.5168636791343213, 0.6009308510638298, 0.4522576934138625, 0.5006751396695192, 0.5168636791343213]


In [31]:
import gc
del mval
del phenotype_df
del combined_df
gc.collect()

300